# Stage 3: Embed Each Step into a 768-Dimensional Vector

Converts each step text into a numerical vector using the `all-mpnet-base-v2` sentence transformer. Two steps with similar meaning will have vectors close together in space. Two steps about different things will be far apart.

These vectors become the node features when we build graphs in Stage 4. The model downloads once (~420MB) and is cached locally after that.

**Input:** `data/gsm8k_steps.csv`  
**Output:** `data/gsm8k_embeddings.npy`, `data/gsm8k_steps_meta.csv`

## Cell 1 — Install libraries

We need one new library: `sentence-transformers`  
It downloads and runs the embedding model locally on your machine.

After this cell → **Kernel → Restart** → run all cells top to bottom.

In [1]:
import sys

# Only run this once in a fresh Anaconda reasoning_graphs environment
# torch 2.5.1 is required — do NOT downgrade to 2.0.1
!{sys.executable} -m pip install torch==2.5.1 --index-url https://download.pytorch.org/whl/cpu -q
!{sys.executable} -m pip install sentence-transformers pandas numpy -q

print(" All libraries installed")
print("   → Kernel → Restart → run from Cell 2")

 All libraries installed
   → Kernel → Restart → run from Cell 2


## Cell 2 — Imports and file paths

We import all libraries and define the input/output file paths.

- `INPUT_FILE` — steps CSV from Stage 2
- `OUTPUT_NPY` — embeddings saved as a numpy array `(1450, 768)` — used by Stage 4 and 5
- `OUTPUT_CSV` — same steps CSV but with 768 extra columns added — useful for inspection

In [2]:
import os
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# File paths
DATA_DIR    = os.path.join(os.getcwd(), 'data')
INPUT_FILE  = os.path.join(DATA_DIR, 'gsm8k_steps.csv')
OUTPUT_NPY  = os.path.join(DATA_DIR, 'gsm8k_embeddings.npy')
OUTPUT_CSV  = os.path.join(DATA_DIR, 'gsm8k_steps_with_embeddings.csv')

print(f" Imports OK")
print(f" Input:      {INPUT_FILE}")
print(f" Output NPY: {OUTPUT_NPY}")
print(f" Output CSV: {OUTPUT_CSV}")

# Check input file exists
if os.path.exists(INPUT_FILE):
    print(f"\n Input file found")
else:
    print(f"\n Input file NOT found — run Stage 2 first")

 Imports OK
 Input:      C:\Users\Aruna\Desktop\latthika_research\cot_project\data\gsm8k_steps.csv
 Output NPY: C:\Users\Aruna\Desktop\latthika_research\cot_project\data\gsm8k_embeddings.npy
 Output CSV: C:\Users\Aruna\Desktop\latthika_research\cot_project\data\gsm8k_steps_with_embeddings.csv

 Input file found


## Cell 3 — Load the steps data

Load `gsm8k_steps.csv` from Stage 2 and print a summary.

We only need the `step_text` column for embedding — but we keep all columns so we can save the full enriched CSV at the end.

In [3]:
df = pd.read_csv(INPUT_FILE)

print(f"Total steps:     {len(df)}")
print(f"Unique traces:   {df['trace_id'].nunique()}")
print(f"Columns:         {list(df.columns)}")
print()

# Split breakdown
print("Steps per split:")
for s in ['train', 'val', 'test']:
    print(f"  {s}: {len(df[df['split']==s])}")
print()

# Label breakdown
counts = df['label'].value_counts()
print(f"Label breakdown:")
print(f"  From correct traces (1): {counts.get(1,0)} steps")
print(f"  From wrong traces   (0): {counts.get(0,0)} steps")
print()

# Show 3 sample step texts
print("Sample step texts:")
for i, row in df.head(3).iterrows():
    print(f"  [{i}] {row['step_text'][:90]}")

print("\n Data loaded")

Total steps:     9490
Unique traces:   1948
Columns:         ['trace_id', 'orig_idx', 'split', 'question', 'step_num', 'step_text', 'total_steps', 'label']

Steps per split:
  train: 6614
  val: 1407
  test: 1469

Label breakdown:
  From correct traces (1): 6400 steps
  From wrong traces   (0): 3090 steps

Sample step texts:
  [0] First, let's find out how many blankets Nathan added to his bed. Since he added half of th
  [1] Each blanket warms up Nathan by 3 degrees. To find the total temperature increase, we mult
  [2] Determine the total amount of ground beef Maurice has purchased.
Maurice bought 4 packages

 Data loaded


## Cell 4 — Load the embedding model

We load `all-mpnet-base-v2` from HuggingFace.  

**First run:** downloads ~420MB — takes 1–2 minutes depending on internet speed.  
**After first run:** loads from local cache instantly — no internet needed.

We also run a quick test to confirm the model outputs 768 numbers per step.

In [4]:
print("Loading all-mpnet-base-v2...")
print("(First run downloads ~420MB — subsequent runs load from cache instantly)")
print()

model = SentenceTransformer('all-mpnet-base-v2')

print(f" Model loaded")
print()

# Quick test — embed one sentence and check output shape
test_embedding = model.encode("Half of 48 is 24 clips.")
print(f"Test embedding shape: {test_embedding.shape}")
print(f"First 5 values:       {test_embedding[:5].round(4)}")
print()

if test_embedding.shape[0] == 768:
    print(" Model outputs 768-dim vectors — correct")
else:
    print(f"  Unexpected output size: {test_embedding.shape[0]}")

Loading all-mpnet-base-v2...
(First run downloads ~420MB — subsequent runs load from cache instantly)

 Model loaded

Test embedding shape: (768,)
First 5 values:       [-0.0318 -0.0899  0.0084  0.0375 -0.0318]

 Model outputs 768-dim vectors — correct


## Cell 5 — Generate embeddings for all steps

This is the main step. We pass all 1,450 step texts through the model in **batches of 64**.

Batching means we process 64 steps at a time instead of one by one — much faster.

**Output:** a numpy array of shape `(1450, 768)` — 1,450 steps × 768 numbers each.

**Expected time: ~5–10 minutes on CPU.**

In [5]:
import time

step_texts = df['step_text'].tolist()

print(f"Embedding {len(step_texts)} steps...")
print(f"Batch size: 64")
print(f"Expected time: ~5–10 minutes on CPU")
print()

start = time.time()

embeddings = model.encode(
    step_texts,
    batch_size=64,
    show_progress_bar=True,    # shows a progress bar
    convert_to_numpy=True      # output as numpy array
)

elapsed = time.time() - start

print()
print(f" Done in {elapsed/60:.1f} minutes")
print(f"   Embeddings shape: {embeddings.shape}")
print(f"   Expected shape:   ({len(step_texts)}, 768)")

if embeddings.shape == (len(step_texts), 768):
    print("    Shape correct")
else:
    print(f"     Unexpected shape — expected ({len(step_texts)}, 768)")

Embedding 9490 steps...
Batch size: 64
Expected time: ~5–10 minutes on CPU



Batches:   0%|          | 0/149 [00:00<?, ?it/s]


 Done in 20.6 minutes
   Embeddings shape: (9490, 768)
   Expected shape:   (9490, 768)
    Shape correct


## Cell 6 — Save the embeddings

We save in two formats:

**1. `gsm8k_embeddings.npy`** — numpy binary format  
This is what Stage 4 and Stage 5 will load directly. Fast to load, compact file size.

**2. `gsm8k_steps_with_embeddings.csv`** — CSV with 768 extra columns (`emb_0` to `emb_767`)  
Useful if you want to inspect or debug the embeddings in Excel.

We also save a **metadata CSV** (`gsm8k_steps_meta.csv`) — the steps file WITHOUT the embedding columns. Stage 4 loads this for trace_id, label, split info without having to load 768 extra columns.

In [6]:
#  Save 1: numpy array (main output for Stage 4+5) 
np.save(OUTPUT_NPY, embeddings)
print(f" Saved embeddings array: {OUTPUT_NPY}")
print(f"   Shape: {embeddings.shape}  |  Size: {os.path.getsize(OUTPUT_NPY)/1e6:.1f} MB")
print()

#  Save 2: metadata CSV (steps info without embedding columns) 
META_CSV = os.path.join(DATA_DIR, 'gsm8k_steps_meta.csv')
df.to_csv(META_CSV, index=False)
print(f" Saved metadata CSV: {META_CSV}")
print(f"   Rows: {len(df)}  |  Columns: {list(df.columns)}")
print()

#  Save 3: full CSV with embedding columns (for inspection) 
emb_cols = pd.DataFrame(
    embeddings,
    columns=[f'emb_{i}' for i in range(embeddings.shape[1])]
)
df_full = pd.concat([df.reset_index(drop=True), emb_cols], axis=1)
df_full.to_csv(OUTPUT_CSV, index=False)
print(f" Saved full CSV: {OUTPUT_CSV}")
print(f"   Rows: {len(df_full)}  |  Total columns: {len(df_full.columns)} (8 meta + 768 emb)")
print()
print("All files saved to data/ folder.")

 Saved embeddings array: C:\Users\Aruna\Desktop\latthika_research\cot_project\data\gsm8k_embeddings.npy
   Shape: (9490, 768)  |  Size: 29.2 MB

 Saved metadata CSV: C:\Users\Aruna\Desktop\latthika_research\cot_project\data\gsm8k_steps_meta.csv
   Rows: 9490  |  Columns: ['trace_id', 'orig_idx', 'split', 'question', 'step_num', 'step_text', 'total_steps', 'label']

 Saved full CSV: C:\Users\Aruna\Desktop\latthika_research\cot_project\data\gsm8k_steps_with_embeddings.csv
   Rows: 9490  |  Total columns: 776 (8 meta + 768 emb)

All files saved to data/ folder.


## Cell 7 — Quality checks

Six checks to confirm embeddings are correct before Stage 4.

| # | Check | Target |
|---|---|---|
| 1 | Embedding shape | (n_steps, 768) |
| 2 | No NaN values | 0 NaNs |
| 3 | No zero vectors | 0 all-zero rows |
| 4 | Value range sensible | between -5 and +5 |
| 5 | Similar steps have similar embeddings | cosine > 0.7 |
| 6 | Different steps have different embeddings | cosine < 0.5 |

In [7]:
from numpy.linalg import norm

# Reload to confirm saved correctly
emb_loaded = np.load(OUTPUT_NPY)
meta       = pd.read_csv(META_CSV)

print("=" * 55)
print("  STAGE 3 QUALITY CHECKS")
print("=" * 55)

all_pass = True

# Check 1: Shape
print(f"\n[1] Embedding shape: {emb_loaded.shape}")
if emb_loaded.shape[0] == len(meta) and emb_loaded.shape[1] == 768:
    print("     PASS")
else:
    print(f"     FAIL — expected ({len(meta)}, 768)")
    all_pass = False

# Check 2: No NaNs
nan_count = np.isnan(emb_loaded).sum()
print(f"\n[2] NaN values: {nan_count}")
if nan_count == 0:
    print("     PASS")
else:
    print(f"     FAIL — {nan_count} NaN values found")
    all_pass = False

# Check 3: No zero vectors
zero_rows = (np.abs(emb_loaded).sum(axis=1) == 0).sum()
print(f"\n[3] Zero vectors: {zero_rows}")
if zero_rows == 0:
    print("     PASS")
else:
    print(f"      {zero_rows} zero vectors found")
    all_pass = False

# Check 4: Value range
vmin = emb_loaded.min()
vmax = emb_loaded.max()
print(f"\n[4] Value range: [{vmin:.3f}, {vmax:.3f}]")
if -5 <= vmin and vmax <= 5:
    print("     PASS — values in normal range")
else:
    print("      Unusual value range")
    all_pass = False

# Check 5: Similar sentences → similar embeddings
def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

sim_text_a = "Half of 48 is 24"
sim_text_b = "48 divided by 2 equals 24"
emb_a = model.encode(sim_text_a)
emb_b = model.encode(sim_text_b)
sim_score = cosine_sim(emb_a, emb_b)
print(f"\n[5] Cosine similarity of similar sentences: {sim_score:.3f}")
print(f"    '{sim_text_a}'")
print(f"    '{sim_text_b}'")
if sim_score > 0.7:
    print("     PASS — similar sentences have similar embeddings")
else:
    print("      Similar sentences not close enough")
    all_pass = False

# Check 6: Different sentences → different embeddings
diff_text_a = "Half of 48 is 24"
diff_text_b = "She went to the supermarket to buy apples"
emb_c = model.encode(diff_text_a)
emb_d = model.encode(diff_text_b)
diff_score = cosine_sim(emb_c, emb_d)
print(f"\n[6] Cosine similarity of different sentences: {diff_score:.3f}")
print(f"    '{diff_text_a}'")
print(f"    '{diff_text_b}'")
if diff_score < 0.5:
    print("     PASS — different sentences have different embeddings")
else:
    print("      Different sentences too similar")
    all_pass = False

print()
print("=" * 55)
if all_pass:
    print("   ALL CHECKS PASSED — Stage 3 complete!")
    print("    Ready for Stage 4 (build graphs)")
else:
    print("    SOME CHECKS FAILED — see above")
print("=" * 55)

  STAGE 3 QUALITY CHECKS

[1] Embedding shape: (9490, 768)
     PASS

[2] NaN values: 0
     PASS

[3] Zero vectors: 0
     PASS

[4] Value range: [-0.225, 0.213]
     PASS — values in normal range

[5] Cosine similarity of similar sentences: 0.868
    'Half of 48 is 24'
    '48 divided by 2 equals 24'
     PASS — similar sentences have similar embeddings

[6] Cosine similarity of different sentences: 0.068
    'Half of 48 is 24'
    'She went to the supermarket to buy apples'
     PASS — different sentences have different embeddings

   ALL CHECKS PASSED — Stage 3 complete!
    Ready for Stage 4 (build graphs)


## Cell 8 — Visual check: are similar steps actually close?

We pick 3 steps from the same trace and 3 steps from different traces and compare their cosine similarity.

**Expected:**
- Steps from the same trace → higher similarity (they're about the same problem)
- Steps from different traces → lower similarity

In [8]:
meta = pd.read_csv(META_CSV)
emb  = np.load(OUTPUT_NPY)

def cosine_sim(a, b):
    return float(np.dot(a, b) / (norm(a) * norm(b)))

# Pick first trace with at least 3 steps
trace_ids = meta.groupby('trace_id').filter(lambda x: len(x) >= 3)['trace_id'].unique()
tid = trace_ids[0]
same_trace = meta[meta['trace_id'] == tid].sort_values('step_num')

print("=" * 65)
print(f"Steps from the SAME trace (trace_id={tid}, label={same_trace.iloc[0]['label']}):")
print("=" * 65)
for _, r in same_trace.head(3).iterrows():
    print(f"  Step {r['step_num']}: {r['step_text'][:70]}")
print()

idx0 = same_trace.index[0]
idx1 = same_trace.index[1]
idx2 = same_trace.index[2]

sim_01 = cosine_sim(emb[idx0], emb[idx1])
sim_02 = cosine_sim(emb[idx0], emb[idx2])
sim_12 = cosine_sim(emb[idx1], emb[idx2])
print(f"  Similarity Step1 vs Step2: {sim_01:.3f}")
print(f"  Similarity Step1 vs Step3: {sim_02:.3f}")
print(f"  Similarity Step2 vs Step3: {sim_12:.3f}")
print()

# Pick steps from 3 completely different traces
diff_traces = meta[meta['trace_id'] != tid]['trace_id'].unique()[:3]
diff_idx    = [meta[meta['trace_id']==t].index[0] for t in diff_traces]

print("=" * 65)
print("Steps from DIFFERENT traces:")
print("=" * 65)
for i, idx in enumerate(diff_idx):
    r = meta.iloc[idx]
    print(f"  Trace {diff_traces[i]}: {r['step_text'][:70]}")
print()

sim_d01 = cosine_sim(emb[diff_idx[0]], emb[diff_idx[1]])
sim_d02 = cosine_sim(emb[diff_idx[0]], emb[diff_idx[2]])
print(f"  Similarity diff_trace_1 vs diff_trace_2: {sim_d01:.3f}")
print(f"  Similarity diff_trace_1 vs diff_trace_3: {sim_d02:.3f}")
print()
print("Same-trace steps should generally score higher than different-trace steps.")

Steps from the SAME trace (trace_id=1, label=1):
  Step 1: Determine the total amount of ground beef Maurice has purchased.
Mauri
  Step 2: Calculate how many 2-pound burgers Maurice can make with the ground be
  Step 3: Since Maurice also wants a burger, subtract 1 from the total number of

  Similarity Step1 vs Step2: 0.778
  Similarity Step1 vs Step3: 0.530
  Similarity Step2 vs Step3: 0.681

Steps from DIFFERENT traces:
  Trace 0: First, let's find out how many blankets Nathan added to his bed. Since
  Trace 2: Let's define the total prize money as 'x'. Rica got 3/8 of the prize m
  Trace 3: We know Phoebe's current age is 10 years old.

  Similarity diff_trace_1 vs diff_trace_2: 0.308
  Similarity diff_trace_1 vs diff_trace_3: 0.196

Same-trace steps should generally score higher than different-trace steps.


##  Stage 3 Complete!

**Output files in `data/` folder:**

| File | What it contains | Used by |
|---|---|---|
| `gsm8k_embeddings.npy` | Numpy array shape (n_steps, 768) | Stage 4, Stage 5 |
| `gsm8k_steps_meta.csv` | Steps info — trace_id, step_num, label, split | Stage 4, Stage 5 |
| `gsm8k_steps_with_embeddings.csv` | Full CSV with 768 embedding columns | Inspection only |

---

**What just happened:**  
Every step text → fed through `all-mpnet-base-v2` → 768 numbers  
Similar steps → similar numbers (high cosine similarity)  
Different steps → different numbers (low cosine similarity)  

**Next → Stage 4:**  
For each trace, connect its steps into a **graph** using 3 edge types:  
- Sequential edges (Step 1 → Step 2 → Step 3)
- Semantic edges (if two steps are about the same thing, cosine > 0.75)
- Value-reuse edges (if a number from Step 2 appears again in Step 4)